# Find bounding box for data

#### Project bounding box for all data:

| boundary | value | 
|----------|-------|
| West longitude | -85.94712712079293 |
| East longitide | -85.3443621648922 |
| South_latitude | 37.99712528351634 |
| North_latitude | 38.38023822809115 |

#### Span

| | difference | approx span | 
|-|------------|-------------|
| ∆ longitude | 0.6027649559007244 | 30 miles |
| ∆ latitude | 0.3831129445748118 | 26 miles |

\* Code for finding the boundaries begins below.

In [1]:
from os import path

import json
import pandas as pd

HOME = "~/code/county_coverage/"


In [2]:
# general functions

def get_data(filepath) -> dict:
    fp = path.join(path.expanduser(HOME), filepath)
    with open(fp, 'r') as data:
        json_data = json.load(data)
    return json_data


def sort_long_lat(points:list) -> pd.Series:
    """Create bounding box from a list of (longitude, latitude) points."""
    longitudes = set()
    latitudes = set()
    for long, lat in points:
        longitudes.add(long)
        latitudes.add(lat)

    return {"west_longitude": min(longitudes), "east_longitude": max(longitudes),
            "south_latitude": min(latitudes), "north_latitude": max(latitudes)}


Source data for county boundary is a GeoJSON file that looks like this:

```json
{
"type": "FeatureCollection",
"name": "Louisville_Metro_KY_County_Boundaries",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
"features": [...]}
```

Where each item in `"features"` represent a county in the Louisville, KY metro area. This includes Jefferson County, where Louisville is, and several surrounding counties. Each `feature` or county object looks like this:

```json
{ "type": "Feature", 
  "properties": { "OBJECTID": 7, 
                  "CNTY_NAME": "JEFFERSON", 
                  "FIPS": "21111", 
                  "STATE_FIPS": "21", 
                  "CNTY_FIPS": "111", 
                  "SHAPEAREA": 11083783720.6446, 
                  "SHAPELEN": 513054.30366378697 }, 
  "geometry": { "type": "Polygon", 
                "coordinates": [ [ [ -85.575811868593064, 38.334545868470848 ], 
                                   [ -85.578070603648868, 38.335714204871564 ], 
                                   [ -85.579003771392806, 38.336193264774884 ],
                                    ... ] ] } }
```

Jefferson County is the one we are interested in here. The value for `"coordinates"` is a list of lists. Each interior list contains points defining a polygon that represents the boundary of the county. The points are listed as `[longitude, latitude]` pairs. The `coordinates` are really all we need, once we find the correct county object in the list of `features`.

In [32]:
METRO_BOUNDARIES = "data/raw/Louisville_Metro_KY_County_Boundaries.geojson"


def get_JEFFCO_data(path_to_data):
    data = get_data(path_to_data)
    counties = data['features']
    # Each feature represents a county boundary. Need to find the right one.
    # The county I am interested in is called Jefferson.

    for county in counties:
        if county['properties']['CNTY_NAME'] == 'JEFFERSON':
            JEFFCO = county
            break
    return JEFFCO

JEFFCO = get_JEFFCO_data(METRO_BOUNDARIES)

from math import sqrt

sqrt(JEFFCO['properties']['SHAPEAREA'])/5280


19.939308777281514

In [18]:

def get_JEFFCO_box(JEFFCO):
    points = JEFFCO['geometry']['coordinates'][0]
                                            # Sometimes Polygon geometry can consist of more than one shape.
                                            # In this case, the county boundary is just one shape.
    return pd.Series(sort_long_lat(points), name='official boundary')

boundaries = pd.DataFrame(get_JEFFCO_box(JEFFCO))
boundaries

,official boundary
west_longitude,-85.947127
east_longitude,-85.404922
south_latitude,37.997125
north_latitude,38.380238


In [4]:
# Both centerlines and intersection GeoJSON data contain an array of features, each of which has properties and geometry
# We are only concerned about the geometry, which has two properties, `type` and `coordinates`. `type` is not particularly helpful here. 

def get_geometries(json_data):
    """Pull all geometry objects out of GeoJSON, ignoring other data."""
    for feature in json_data['features']:
        yield feature['geometry']['coordinates']

In [5]:
# Get bounding box for centerlines
CENTERLINES = "data/raw/centerlines/Jefferson_County_KY_Street_Centerlines.geojson"

centerline_geometries = get_geometries(get_data(CENTERLINES))
centerline_points = (point for geometry in centerline_geometries for point in geometry)

boundaries['centerlines'] = sort_long_lat(centerline_points)
#boundaries

In [6]:
# Get bounding box for intersections

# Add bounding box from intersection metadata

INT_LL_BOX = {"west_longitude": -85.945347, "east_longitude": -85.344499,
              "north_latitude": 38.378034, "south_latitude": 38.005894}

boundaries['intersection metadata'] = INT_LL_BOX
#boundaries

In [7]:
# TODO convert this

#Extent in the item's coordinate system 
west_longitude_co = 1154395.500000
east_longitude_co = 1325086.990000
south_latitude_co = 188677.437500
north_latitude_co = 321629.781250
#* Extent contains the resource Yes

In [8]:
# Derive box from intersection data

INTERSECTIONS = "data/raw/intersections/Jefferson_County_KY_Street_Intersections.geojson"
intersection_data = get_data(INTERSECTIONS)
intersection_points = get_geometries(intersection_data)

boundaries['intersections derived'] = sort_long_lat(intersection_points)
#boundaries


In [9]:
# compare boundary values to find the largest bounding box that covers all the data

BT = boundaries.T

comparisons = {'west_longitude': BT.west_longitude.min(), 'east_longitude': BT.east_longitude.max(),
               'south_latitude': BT.south_latitude.min(), 'north_latitude': BT.north_latitude.max()}

boundaries['comparison'] = comparisons
display(boundaries)
#boundaries.comparison.to_dict()

,official boundary,centerlines,intersection metadata,intersections derived,comparison
west_longitude,-85.947127,-85.942405,-85.945347,-85.936859,-85.947127
east_longitude,-85.404922,-85.344362,-85.344499,-85.347451,-85.344362
south_latitude,37.997125,38.000584,38.005894,38.005901,37.997125
north_latitude,38.380238,38.377076,38.378034,38.375609,38.380238


### Conclusion:

Official county boundary covers almost everything, except for `east_longitude` (maximum longitude), where some of the centerlines must have points east of the official county boundary. I also know that some of the intersections lie outside of Jefferson county.

The largest bounding box is represented by the `comparison` column in the last dataframe. This bounding box covers all the data across all the different input files. 

In [11]:
comps = boundaries.comparison
display(comps.to_dict())

longitude_delta = comps.east_longitude - comps.west_longitude
latitude_delta = comps.north_latitude - comps.south_latitude
longitude_delta, latitude_delta


{'west_longitude': -85.94712712079293,
 'east_longitude': -85.3443621648922,
 'south_latitude': 37.99712528351634,
 'north_latitude': 38.38023822809115}

(np.float64(0.6027649559007244), np.float64(0.3831129445748118))

In [17]:
from common.county_geometry import *

east_point = (east_longitude, north_latitude)
west_point = (west_longitude, north_latitude)

long_mi = haversine_distance_mi(east_point, west_point)
long_km = haversine_distance_km(east_point, west_point)

display(f"longitude span = {long_mi} miles == {long_km} kilometers.")


north_point = (west_longitude, north_latitude)
south_point = (west_longitude, south_latitude)

lat_mi = haversine_distance_mi(south_point, north_point)
lat_km = haversine_distance_km(south_point, north_point)

display(f'latitude span = {lat_mi} miles == {lat_km} kilometers.')

'longitude span = 32.62464358670343 miles == 52.50786292126914 kilometers.'

'latitude span = 26.45211953861136 miles == 42.57346943941823 kilometers.'